# Этап 5. Эксперименты V3

Пошаговая проверка поиска V3 на тех же 720 вопросах и qrels, что и в V2. Метрики считает код V2, поэтому числа сравнимы с V2.

В ноутбуке нет заранее вписанных чисел: таблица и графики строятся только из файлов `results/<запуск>/summary_all.json`, которые создаёт `scripts/v3_evaluation.py` (или `scripts/run_all.ps1`).

In [ ]:
import sys, json
from pathlib import Path
V3 = Path.cwd()
while not (V3 / "configs/v3_config.json").is_file():  # ноутбук лежит в V3/docs/notebooks
    if V3 == V3.parent:
        raise FileNotFoundError("Не найдена папка V3 (configs/v3_config.json)")
    V3 = V3.parent
for p in (V3 / "src", V3 / "scripts"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
import plotly.io as pio
pio.renderers.default = "plotly_mimetype+notebook_connected"  # графики видны и после сохранения

## 1. Запуск оценки

Первый запуск занимает время (E5, CLIP и BGE на GPU). Если оценка уже выполнена, ячейку можно пропустить: ниже берётся последний запуск.

In [ ]:
RUN_EVALUATION = False
if RUN_EVALUATION:
    import v3_evaluation
    v3_evaluation.main([])

## 2. Сводная таблица

In [ ]:
import pandas as pd
runs = sorted(p for p in (V3 / "results").glob("*/summary_all.json"))
assert runs, "Нет результатов: запустите scripts/run_all.ps1"
latest = runs[-1]
print("Запуск:", latest.parent.name)
df = pd.DataFrame(json.loads(latest.read_text(encoding="utf-8")))
display(df.style.format({c: "{:.4f}" for c in df.columns if c not in ("Шаг", "questions")}, decimal=","))

## 3. Сравнение с V2

В список `V2_METRICS` добавляются файлы `metrics.json` прогонов V2 на тех же вопросах (путь от папки V3, например `../V2/results/...`).

In [ ]:
from v3_evaluation import summarize
V2_METRICS = {
    # "V2 опыт 2 (E5 + BGE)": "../V2/results/retrieval/split/<запуск>/metrics.json",
}
rows = []
for name, path in V2_METRICS.items():
    rows.append({"Шаг": name, **summarize(json.loads((V3 / path).read_text(encoding="utf-8")), "all")})
compare = pd.concat([pd.DataFrame(rows), df], ignore_index=True) if rows else df
display(compare)

## 4. Графики

In [ ]:
import plotly.graph_objects as go
fig = go.Figure()
for k in ("Hit@1", "Hit@3", "Hit@5", "Hit@10"):
    fig.add_bar(name=k, x=compare["Шаг"], y=compare[k])
fig.update_layout(title="Hit@k по шагам", barmode="group", yaxis_range=[0, 1], separators=", ")
fig.show()

fig = go.Figure()
for m in ("MRR@10", "nDCG@10"):
    fig.add_scatter(name=m, x=compare["Шаг"], y=compare[m], mode="lines+markers")
fig.update_layout(title="MRR@10 и nDCG@10 по шагам", yaxis_range=[0, 1], separators=", ")
fig.show()

## 5. Прирост каждого шага относительно предыдущего, п.п.

In [ ]:
delta = df.set_index("Шаг")[["Hit@1", "Hit@5", "MRR@10"]].diff().mul(100).round(1)
display(delta)